# Coding Practice Session 13
## Grouping and Aggregating Data

In [1]:
import pandas as pd
import numpy as np

In [2]:
np.random.seed(25)

In [3]:
dates = pd.date_range(start="2024-01-01", end="2025-12-31", freq="D")
products = ["Laptop", "Smartphone", "Tablet", "Headphones"]
regions = ["North", "South", "East", "West"]

In [4]:
num_entries = 1000

In [5]:
df = pd.DataFrame(
    {
        "Date": np.random.choice(dates, num_entries),
        "Product": np.random.choice(products, num_entries),
        "Region": np.random.choice(regions, num_entries),
        "Sales": np.random.randint(100, 1500, num_entries),
        "Units": np.random.randint(1, 10, num_entries),
    }
)

In [6]:
df

,Date,Product,Region,Sales,Units
0,2024-05-12,Laptop,East,1285,8
1,2024-11-14,Smartphone,South,1411,8
2,2025-04-19,Tablet,East,1429,2
3,2024-05-23,Smartphone,North,962,2
4,2024-11-13,Tablet,West,775,4
...,...,...,...,...,...
995,2025-11-19,Headphones,North,1140,1
996,2025-07-15,Tablet,West,599,3
997,2024-11-04,Laptop,East,827,7
998,2024-07-03,Laptop,East,432,4


In [7]:
df["Revenue"] = df["Sales"] * df["Units"]

In [8]:
df

,Date,Product,Region,Sales,Units,Revenue
0,2024-05-12,Laptop,East,1285,8,10280
1,2024-11-14,Smartphone,South,1411,8,11288
2,2025-04-19,Tablet,East,1429,2,2858
3,2024-05-23,Smartphone,North,962,2,1924
4,2024-11-13,Tablet,West,775,4,3100
...,...,...,...,...,...,...
995,2025-11-19,Headphones,North,1140,1,1140
996,2025-07-15,Tablet,West,599,3,1797
997,2024-11-04,Laptop,East,827,7,5789
998,2024-07-03,Laptop,East,432,4,1728


In [9]:
df = df.sort_values("Date")

In [10]:
df.reset_index(drop=True, inplace=True)

In [11]:
df

,Date,Product,Region,Sales,Units,Revenue
0,2024-01-01,Headphones,North,713,9,6417
1,2024-01-01,Smartphone,South,866,6,5196
2,2024-01-02,Tablet,North,1038,8,8304
3,2024-01-02,Tablet,North,1305,5,6525
4,2024-01-03,Smartphone,East,523,5,2615
...,...,...,...,...,...,...
995,2025-12-28,Headphones,North,1224,2,2448
996,2025-12-29,Smartphone,North,448,2,896
997,2025-12-29,Tablet,South,336,7,2352
998,2025-12-30,Tablet,North,621,7,4347


In [12]:
df["Product"] = df["Product"].astype("category")

In [13]:
df["Product"].value_counts()

Product
Laptop        259
Headphones    257
Tablet        249
Smartphone    235
Name: count, dtype: int64

### Basic Grouping with `groupby()`

In [14]:
df.groupby("Product", observed=True)["Revenue"].sum()

Product
Headphones     958286
Laptop        1033886
Smartphone     882313
Tablet         963400
Name: Revenue, dtype: int64

In [15]:
df.groupby("Product", observed=True)["Revenue"].sum().sort_values(ascending=False)

Product
Laptop        1033886
Tablet         963400
Headphones     958286
Smartphone     882313
Name: Revenue, dtype: int64

In [16]:
df.groupby("Product", observed=True).agg(
    {"Revenue": "sum", "Units": "mean", "Sales": "mean"}
)

,Revenue,Units,Sales
Product,,,
Headphones,958286,4.782101,778.381323
Laptop,1033886,5.027027,784.644788
Smartphone,882313,4.889362,780.148936
Tablet,963400,4.895582,794.261044


In [17]:
df.groupby(["Region", "Product"], observed=True)["Revenue"].sum()

Region  Product   
East    Headphones    262419
        Laptop        258383
        Smartphone    226576
        Tablet        232959
North   Headphones    264924
        Laptop        269503
        Smartphone    261388
        Tablet        208650
South   Headphones    196608
        Laptop        226668
        Smartphone    213503
        Tablet        298257
West    Headphones    234335
        Laptop        279332
        Smartphone    180846
        Tablet        223534
Name: Revenue, dtype: int64

In [18]:
df.groupby(["Region", "Product"], observed=True)["Revenue"].sum().unstack()

Product,Headphones,Laptop,Smartphone,Tablet
Region,,,,
East,262419,258383,226576,232959
North,264924,269503,261388,208650
South,196608,226668,213503,298257
West,234335,279332,180846,223534


In [19]:
df.groupby(["Region", "Product"], observed=True)["Revenue"].sum().unstack(
    level="Region"
)

Region,East,North,South,West
Product,,,,
Headphones,262419,264924,196608,234335
Laptop,258383,269503,226668,279332
Smartphone,226576,261388,213503,180846
Tablet,232959,208650,298257,223534


In [20]:
def get_quarter(date):
    return f"Q{date.quarter}"

In [21]:
df.groupby(df["Date"].apply(get_quarter))["Revenue"].sum().sort_values(ascending=False)

Date
Q1    1017268
Q3     974240
Q2     949444
Q4     896933
Name: Revenue, dtype: int64

In [22]:
df.groupby(df["Date"].dt.to_period("M"))["Revenue"].sum()

Date
2024-01    185023
2024-02    169187
2024-03    140064
2024-04    120656
2024-05    183941
2024-06    132474
2024-07    163530
2024-08    183507
2024-09    167244
2024-10    130568
2024-11    179595
2024-12    143637
2025-01    203575
2025-02    125490
2025-03    193929
2025-04    181071
2025-05    173034
2025-06    158268
2025-07    168438
2025-08    180347
2025-09    111174
2025-10    138363
2025-11    142374
2025-12    162396
Freq: M, Name: Revenue, dtype: int64

In [23]:
df.groupby(df["Date"].dt.to_period("Q"))["Revenue"].sum()

Date
2024Q1    494274
2024Q2    437071
2024Q3    514281
2024Q4    453800
2025Q1    522994
2025Q2    512373
2025Q3    459959
2025Q4    443133
Freq: Q-DEC, Name: Revenue, dtype: int64

In [24]:
df.groupby([df["Date"].apply(get_quarter), "Product"], observed=True)[
    "Revenue"
].sum().unstack(level="Product")

Product,Headphones,Laptop,Smartphone,Tablet
Date,,,,
Q1,242842,219660,284904,269862
Q2,198429,273864,200477,276674
Q3,289559,296361,190901,197419
Q4,227456,244001,206031,219445


### Aggregation Methods

In [25]:
df

,Date,Product,Region,Sales,Units,Revenue
0,2024-01-01,Headphones,North,713,9,6417
1,2024-01-01,Smartphone,South,866,6,5196
2,2024-01-02,Tablet,North,1038,8,8304
3,2024-01-02,Tablet,North,1305,5,6525
4,2024-01-03,Smartphone,East,523,5,2615
...,...,...,...,...,...,...
995,2025-12-28,Headphones,North,1224,2,2448
996,2025-12-29,Smartphone,North,448,2,896
997,2025-12-29,Tablet,South,336,7,2352
998,2025-12-30,Tablet,North,621,7,4347


In [26]:
grouped = df.groupby("Product", observed=True)

In [27]:
grouped.agg({"Revenue": ["count", "sum", "mean", "median"]})

Revenue                              
             count      sum         mean  median
Product                                         
Headphones     257   958286  3728.739300  2734.0
Laptop         259  1033886  3991.837838  3241.0
Smartphone     235   882313  3754.523404  2830.0
Tablet         249   963400  3869.076305  3033.0

In [28]:
grouped.agg({"Revenue": ["sum", "mean"], "Units": ["count", "max", "min"]})

Revenue              Units        
                sum         mean count max min
Product                                       
Headphones   958286  3728.739300   257   9   1
Laptop      1033886  3991.837838   259   9   1
Smartphone   882313  3754.523404   235   9   1
Tablet       963400  3869.076305   249   9   1

In [29]:
def sales_range(column):
    return column.max() - column.min()

In [30]:
grouped.agg(
    {
        "Revenue": ["sum"],
        "Sales": ["max", sales_range],
    }
)

Revenue Sales            
                sum   max sales_range
Product                              
Headphones   958286  1475        1368
Laptop      1033886  1490        1371
Smartphone   882313  1477        1377
Tablet       963400  1498        1393

In [ ]:
grouped["Revenue"].agg(
    [
        ("Total", "sum"),
        ("Average", "mean"),
        ("Range", lambda col: col.max() - col.min()),
    ]
).round(2)

,Total,Average,Range
Product,,,
Headphones,958286,3728.74,13158
Laptop,1033886,3991.84,12316
Smartphone,882313,3754.52,12399
Tablet,963400,3869.08,13267


In [ ]:
df.groupby("Region").agg(
    {
        "Revenue": ["sum", "mean", "median"],
        "Units": ["sum", "mean"],
        "Sales": ["min", "max"],
    }
)

Revenue                      Units           Sales      
            sum         mean  median   sum      mean   min   max
Region                                                          
East     980337  3874.849802  3080.0  1267  5.007905   107  1496
North   1004465  3734.070632  2877.0  1316  4.892193   100  1487
South    935036  3863.785124  2789.5  1184  4.892562   101  1498
West     918047  3890.029661  3077.0  1132  4.796610   117  1487

In [33]:
grouped.agg(
    Total_Revenue=("Revenue", "sum"),
    Average_Price=("Sales", "mean"),
    Total_Units=("Units", "sum"),
)

,Total_Revenue,Average_Price,Total_Units
Product,,,
Headphones,958286,778.381323,1229
Laptop,1033886,784.644788,1302
Smartphone,882313,780.148936,1149
Tablet,963400,794.261044,1219


### Advanced Grouping Techniques

In [34]:
grouped

In [35]:
grouped.get_group("Laptop")

,Date,Product,Region,Sales,Units,Revenue
5,2024-01-03,Laptop,East,121,6,726
7,2024-01-05,Laptop,North,623,9,5607
15,2024-01-11,Laptop,West,895,8,7160
18,2024-01-12,Laptop,South,434,8,3472
20,2024-01-14,Laptop,South,376,3,1128
...,...,...,...,...,...,...
982,2025-12-20,Laptop,East,854,2,1708
988,2025-12-22,Laptop,North,346,4,1384
989,2025-12-23,Laptop,North,1307,4,5228
990,2025-12-24,Laptop,North,339,9,3051


In [ ]:
grouped.get_group("Laptop").agg(
    {"Revenue": ["sum", "mean"], "Units": "sum", "Sales": ["min", "max", "mean"]}
)

,Revenue,Units,Sales
sum,1.033886e+06,1302.0,NaN
mean,3.991838e+03,NaN,784.644788
min,NaN,NaN,119.000000
max,NaN,NaN,1490.000000


In [37]:
df["Product"].value_counts()

Product
Laptop        259
Headphones    257
Tablet        249
Smartphone    235
Name: count, dtype: int64

In [ ]:
high_number_groups = grouped.filter(lambda group: len(group) > 250)
high_number_groups

,Date,Product,Region,Sales,Units,Revenue
0,2024-01-01,Headphones,North,713,9,6417
5,2024-01-03,Laptop,East,121,6,726
6,2024-01-04,Headphones,North,187,9,1683
7,2024-01-05,Laptop,North,623,9,5607
13,2024-01-09,Headphones,West,1193,5,5965
...,...,...,...,...,...,...
990,2025-12-24,Laptop,North,339,9,3051
991,2025-12-24,Headphones,East,318,3,954
992,2025-12-25,Laptop,East,1138,5,5690
993,2025-12-26,Headphones,West,903,6,5418


In [39]:
high_number_groups["Product"].unique()

['Headphones', 'Laptop']
Categories (4, object): ['Headphones', 'Laptop', 'Smartphone', 'Tablet']

In [ ]:
high_number_groups.groupby("Product", observed=True).agg(
    {"Revenue": ["sum", "mean"], "Units": "sum"}
).dropna().round(decimals=2)

Revenue          Units
                sum     mean   sum
Product                           
Headphones   958286  3728.74  1229
Laptop      1033886  3991.84  1302

### Transformation and Applying Functions to Groups

In [43]:
df

,Date,Product,Region,Sales,Units,Revenue
0,2024-01-01,Headphones,North,713,9,6417
1,2024-01-01,Smartphone,South,866,6,5196
2,2024-01-02,Tablet,North,1038,8,8304
3,2024-01-02,Tablet,North,1305,5,6525
4,2024-01-03,Smartphone,East,523,5,2615
...,...,...,...,...,...,...
995,2025-12-28,Headphones,North,1224,2,2448
996,2025-12-29,Smartphone,North,448,2,896
997,2025-12-29,Tablet,South,336,7,2352
998,2025-12-30,Tablet,North,621,7,4347


In [ ]:
# using .transform()
df["Revenue Percentage"] = df.groupby("Product", observed=True)["Revenue"].transform(
    lambda x: x / x.sum() * 100
)
df["Revenue Percentage"]

0      0.669633
1      0.588907
2      0.861947
3      0.677289
4      0.296380
         ...   
995    0.255456
996    0.101551
997    0.244135
998    0.451214
999    0.094298
Name: Revenue Percentage, Length: 1000, dtype: float64

In [48]:
df

,Date,Product,Region,Sales,Units,Revenue,Revenue Percentage
0,2024-01-01,Headphones,North,713,9,6417,0.669633
1,2024-01-01,Smartphone,South,866,6,5196,0.588907
2,2024-01-02,Tablet,North,1038,8,8304,0.861947
3,2024-01-02,Tablet,North,1305,5,6525,0.677289
4,2024-01-03,Smartphone,East,523,5,2615,0.296380
...,...,...,...,...,...,...,...
995,2025-12-28,Headphones,North,1224,2,2448,0.255456
996,2025-12-29,Smartphone,North,448,2,896,0.101551
997,2025-12-29,Tablet,South,336,7,2352,0.244135
998,2025-12-30,Tablet,North,621,7,4347,0.451214


In [53]:
df[df["Product"] == "Smartphone"].sort_values(by="Revenue Percentage", ascending=False)

,Date,Product,Region,Sales,Units,Revenue,Revenue Percentage
51,2024-02-02,Smartphone,West,1400,9,12600,1.428065
876,2025-09-28,Smartphone,East,1298,9,11682,1.324020
201,2024-05-30,Smartphone,South,1264,9,11376,1.289338
433,2024-11-14,Smartphone,South,1411,8,11288,1.279365
203,2024-05-31,Smartphone,West,1231,9,11079,1.255677
...,...,...,...,...,...,...,...
43,2024-01-30,Smartphone,North,149,2,298,0.033775
421,2024-11-09,Smartphone,West,129,2,258,0.029241
546,2025-01-27,Smartphone,North,231,1,231,0.026181
892,2025-10-11,Smartphone,South,230,1,230,0.026068


In [49]:
df.groupby("Product", observed=True)["Revenue Percentage"].sum()

Product
Headphones    100.0
Laptop        100.0
Smartphone    100.0
Tablet        100.0
Name: Revenue Percentage, dtype: float64

In [ ]:
df["Revenue_Diff_from_Mean"] = df.groupby("Region")["Revenue"].transform(
    lambda x: x - x.mean()
)

In [55]:
df["Revenue_Diff_from_Mean"]

0      2682.929368
1      1332.214876
2      4569.929368
3      2790.929368
4     -1259.849802
          ...     
995   -1286.070632
996   -2838.070632
997   -1511.785124
998     612.929368
999   -3031.785124
Name: Revenue_Diff_from_Mean, Length: 1000, dtype: float64

In [ ]:
df.sort_values(by="Revenue_Diff_from_Mean")

,Date,Product,Region,Sales,Units,Revenue,Revenue Percentage,Revenue_Diff_from_Mean
956,2025-11-27,Headphones,West,117,1,117,0.012209,-3773.029661
736,2025-06-10,Tablet,South,116,1,116,0.012041,-3747.785124
227,2024-06-20,Headphones,West,174,1,174,0.018157,-3716.029661
100,2024-03-12,Tablet,East,169,1,169,0.017542,-3705.849802
324,2024-08-23,Headphones,West,191,1,191,0.019931,-3699.029661
...,...,...,...,...,...,...,...,...
568,2025-02-12,Laptop,East,1391,9,12519,1.210869,8644.150198
51,2024-02-02,Smartphone,West,1400,9,12600,1.428065,8709.970339
107,2024-03-20,Headphones,East,1472,9,13248,1.382468,9373.150198
358,2024-09-18,Headphones,West,1475,9,13275,1.385286,9384.970339


In [ ]:
df["Sales Rank"] = df.groupby("Product", observed=True)["Sales"].transform(
    "rank", method="dense", ascending=False
)
df

,Date,Product,Region,Sales,Units,Revenue,Revenue Percentage,Revenue_Diff_from_Mean,Sales Rank
0,2024-01-01,Headphones,North,713,9,6417,0.669633,2682.929368,128.0
1,2024-01-01,Smartphone,South,866,6,5196,0.588907,1332.214876,101.0
2,2024-01-02,Tablet,North,1038,8,8304,0.861947,4569.929368,70.0
3,2024-01-02,Tablet,North,1305,5,6525,0.677289,2790.929368,38.0
4,2024-01-03,Smartphone,East,523,5,2615,0.296380,-1259.849802,151.0
...,...,...,...,...,...,...,...,...,...
995,2025-12-28,Headphones,North,1224,2,2448,0.255456,-1286.070632,41.0
996,2025-12-29,Smartphone,North,448,2,896,0.101551,-2838.070632,162.0
997,2025-12-29,Tablet,South,336,7,2352,0.244135,-1511.785124,184.0
998,2025-12-30,Tablet,North,621,7,4347,0.451214,612.929368,140.0


In [75]:
df.groupby("Product", observed=True).get_group("Laptop").sort_values("Sales Rank")

,Date,Product,Region,Sales,Units,Revenue,Revenue Percentage,Revenue_Diff_from_Mean,Sales Rank
676,2025-05-03,Laptop,East,1490,4,5960,0.576466,2085.150198,1.0
104,2024-03-17,Laptop,East,1486,3,4458,0.431189,583.150198,2.0
192,2024-05-24,Laptop,East,1485,6,8910,0.861797,5035.150198,3.0
665,2025-04-26,Laptop,North,1457,2,2914,0.281849,-820.070632,4.0
523,2025-01-17,Laptop,North,1449,1,1449,0.140151,-2285.070632,5.0
...,...,...,...,...,...,...,...,...,...
680,2025-05-06,Laptop,North,144,5,720,0.069640,-3014.070632,235.0
187,2024-05-23,Laptop,North,144,4,576,0.055712,-3158.070632,235.0
785,2025-07-17,Laptop,East,136,9,1224,0.118388,-2650.849802,236.0
5,2024-01-03,Laptop,East,121,6,726,0.070221,-3148.849802,237.0


In [77]:
df[["Product", "Sales", "Sales Rank"]].sort_values("Sales Rank").head(8)

,Product,Sales,Sales Rank
462,Smartphone,1477,1.0
358,Headphones,1475,1.0
676,Laptop,1490,1.0
576,Smartphone,1477,1.0
600,Tablet,1498,1.0
107,Headphones,1472,2.0
104,Laptop,1486,2.0
765,Tablet,1496,2.0


In [ ]:
df["Cumulative Units"] = df.groupby("Product", observed=True)["Units"].transform(
    "cumsum"
)
df[["Date", "Product", "Units", "Cumulative Units"]].head(10)

,Date,Product,Units,Cumulative Units
0,2024-01-01,Headphones,9,9
1,2024-01-01,Smartphone,6,6
2,2024-01-02,Tablet,8,8
3,2024-01-02,Tablet,5,13
4,2024-01-03,Smartphone,5,11
5,2024-01-03,Laptop,6,6
6,2024-01-04,Headphones,9,18
7,2024-01-05,Laptop,9,15
8,2024-01-06,Tablet,7,20
9,2024-01-06,Tablet,5,25


In [83]:
df

,Date,Product,Region,Sales,Units,Revenue,Revenue Percentage,Revenue_Diff_from_Mean,Sales Rank,Cumulative Units
0,2024-01-01,Headphones,North,713,9,6417,0.669633,2682.929368,128.0,9
1,2024-01-01,Smartphone,South,866,6,5196,0.588907,1332.214876,101.0,6
2,2024-01-02,Tablet,North,1038,8,8304,0.861947,4569.929368,70.0,8
3,2024-01-02,Tablet,North,1305,5,6525,0.677289,2790.929368,38.0,13
4,2024-01-03,Smartphone,East,523,5,2615,0.296380,-1259.849802,151.0,11
...,...,...,...,...,...,...,...,...,...,...
995,2025-12-28,Headphones,North,1224,2,2448,0.255456,-1286.070632,41.0,1229
996,2025-12-29,Smartphone,North,448,2,896,0.101551,-2838.070632,162.0,1145
997,2025-12-29,Tablet,South,336,7,2352,0.244135,-1511.785124,184.0,1212
998,2025-12-30,Tablet,North,621,7,4347,0.451214,612.929368,140.0,1219


In [ ]:
# using .apply()
def top_2_sales(group):
    return group.nlargest(2, "Sales")[["Date", "Sales"]]


df.groupby("Product", observed=True)[["Date", "Sales"]].apply(top_2_sales)

Date  Sales
Product                         
Headphones 358 2024-09-18   1475
           107 2024-03-20   1472
Laptop     676 2025-05-03   1490
           104 2024-03-17   1486
Smartphone 462 2024-12-03   1477
           576 2025-02-26   1477
Tablet     600 2025-03-11   1498
           765 2025-07-02   1496

In [89]:
df.loc[765]

Date                      2025-07-02 00:00:00
Product                                Tablet
Region                                   East
Sales                                    1496
Units                                       7
Revenue                                 10472
Revenue Percentage                   1.086984
Revenue_Diff_from_Mean            6597.150198
Sales Rank                                2.0
Cumulative Units                          935
Name: 765, dtype: object

In [90]:
df.groupby("Region")[["Sales", "Units"]].apply(lambda x: x["Sales"].corr(x["Units"]))

Region
East    -0.013096
North   -0.072571
South   -0.014806
West     0.097812
dtype: float64

### Exercise

In [91]:
df

,Date,Product,Region,Sales,Units,Revenue,Revenue Percentage,Revenue_Diff_from_Mean,Sales Rank,Cumulative Units
0,2024-01-01,Headphones,North,713,9,6417,0.669633,2682.929368,128.0,9
1,2024-01-01,Smartphone,South,866,6,5196,0.588907,1332.214876,101.0,6
2,2024-01-02,Tablet,North,1038,8,8304,0.861947,4569.929368,70.0,8
3,2024-01-02,Tablet,North,1305,5,6525,0.677289,2790.929368,38.0,13
4,2024-01-03,Smartphone,East,523,5,2615,0.296380,-1259.849802,151.0,11
...,...,...,...,...,...,...,...,...,...,...
995,2025-12-28,Headphones,North,1224,2,2448,0.255456,-1286.070632,41.0,1229
996,2025-12-29,Smartphone,North,448,2,896,0.101551,-2838.070632,162.0,1145
997,2025-12-29,Tablet,South,336,7,2352,0.244135,-1511.785124,184.0,1212
998,2025-12-30,Tablet,North,621,7,4347,0.451214,612.929368,140.0,1219


In [97]:
monthly_sales = df.groupby(df["Date"].dt.to_period("M"))["Revenue"].sum().reset_index()
monthly_sales

,Date,Revenue
0,2024-01,185023
1,2024-02,169187
2,2024-03,140064
3,2024-04,120656
4,2024-05,183941
5,2024-06,132474
6,2024-07,163530
7,2024-08,183507
8,2024-09,167244
9,2024-10,130568


In [100]:
monthly_sales["Month"] = monthly_sales["Date"].dt.strftime("%Y-%m")

In [101]:
monthly_sales

,Date,Revenue,Month
0,2024-01,185023,2024-01
1,2024-02,169187,2024-02
2,2024-03,140064,2024-03
3,2024-04,120656,2024-04
4,2024-05,183941,2024-05
5,2024-06,132474,2024-06
6,2024-07,163530,2024-07
7,2024-08,183507,2024-08
8,2024-09,167244,2024-09
9,2024-10,130568,2024-10


In [104]:
monthly_sales = monthly_sales.set_index("Month")["Revenue"]

In [105]:
monthly_sales

Month
2024-01    185023
2024-02    169187
2024-03    140064
2024-04    120656
2024-05    183941
2024-06    132474
2024-07    163530
2024-08    183507
2024-09    167244
2024-10    130568
2024-11    179595
2024-12    143637
2025-01    203575
2025-02    125490
2025-03    193929
2025-04    181071
2025-05    173034
2025-06    158268
2025-07    168438
2025-08    180347
2025-09    111174
2025-10    138363
2025-11    142374
2025-12    162396
Name: Revenue, dtype: int64

In [107]:
df.groupby("Product", observed=True)["Revenue"].sum().sort_values(ascending=False)

Product
Laptop        1033886
Tablet         963400
Headphones     958286
Smartphone     882313
Name: Revenue, dtype: int64

In [ ]:
df.groupby("Region").agg(
    {
        "Revenue": "sum",
        "Units": "sum",
        "Sales": "mean",
    }
).sort_values(by="Revenue").round(decimals=0)

,Revenue,Units,Sales
Region,,,
West,918047,1132,791.0
South,935036,1184,793.0
East,980337,1267,777.0
North,1004465,1316,778.0


In [ ]:
df["Season"] = df["Date"].dt.month.map(
    {
        1: "Winter",
        2: "Winter",
        3: "Spring",
        4: "Spring",
        5: "Spring",
        6: "Summer",
        7: "Summer",
        8: "Summer",
        9: "Fall",
        10: "Fall",
        11: "Fall",
        12: "Winter",
    }
)

In [111]:
df

,Date,Product,Region,Sales,Units,Revenue,Revenue Percentage,Revenue_Diff_from_Mean,Sales Rank,Cumulative Units,Season
0,2024-01-01,Headphones,North,713,9,6417,0.669633,2682.929368,128.0,9,Winter
1,2024-01-01,Smartphone,South,866,6,5196,0.588907,1332.214876,101.0,6,Winter
2,2024-01-02,Tablet,North,1038,8,8304,0.861947,4569.929368,70.0,8,Winter
3,2024-01-02,Tablet,North,1305,5,6525,0.677289,2790.929368,38.0,13,Winter
4,2024-01-03,Smartphone,East,523,5,2615,0.296380,-1259.849802,151.0,11,Winter
...,...,...,...,...,...,...,...,...,...,...,...
995,2025-12-28,Headphones,North,1224,2,2448,0.255456,-1286.070632,41.0,1229,Winter
996,2025-12-29,Smartphone,North,448,2,896,0.101551,-2838.070632,162.0,1145,Winter
997,2025-12-29,Tablet,South,336,7,2352,0.244135,-1511.785124,184.0,1212,Winter
998,2025-12-30,Tablet,North,621,7,4347,0.451214,612.929368,140.0,1219,Winter


In [115]:
df.groupby(["Season", "Product"], observed=True)["Revenue"].sum().unstack(level="Product")

Product,Headphones,Laptop,Smartphone,Tablet
Season,,,,
Fall,243490,244368,186064,195396
Spring,220229,239740,230508,302218
Summer,276925,304216,170266,235157
Winter,217642,245562,295475,230629
